### Setup


In [1]:
import pandas as pd
import re

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


### Task1 Load & Understand Dataset


In [4]:
# Load the Kaggle movie dataset
df = pd.read_csv("tmdb_5000_movies.csv")

print("Dataset Shape:")
print(df.shape)

print("\nColumn Names:")
print(df.columns.tolist())

print("\nFirst 5 Rows:")
print(df.head())

print("\nMissing Values:")
print(df[["title", "overview", "genres", "keywords"]].isnull().sum())

# Columns used for the recommendation system
print("\nColumns Used:")
print(["title", "overview", "genres", "keywords"])


Dataset Shape:
(4803, 20)

Column Names:
['budget', 'genres', 'homepage', 'id', 'keywords', 'original_language', 'original_title', 'overview', 'popularity', 'production_companies', 'production_countries', 'release_date', 'revenue', 'runtime', 'spoken_languages', 'status', 'tagline', 'title', 'vote_average', 'vote_count']

First 5 Rows:
      budget                                             genres  \
0  237000000  [{"id": 28, "name": "Action"}, {"id": 12, "nam...   
1  300000000  [{"id": 12, "name": "Adventure"}, {"id": 14, "...   
2  245000000  [{"id": 28, "name": "Action"}, {"id": 12, "nam...   
3  250000000  [{"id": 28, "name": "Action"}, {"id": 80, "nam...   
4  260000000  [{"id": 28, "name": "Action"}, {"id": 12, "nam...   

                                       homepage      id  \
0                   http://www.avatarmovie.com/   19995   
1  http://disney.go.com/disneypictures/pirates/     285   
2   http://www.sonypictures.com/movies/spectre/  206647   
3            http://www

### Task2 Text Preprocessing


In [5]:
# Replace missing text with an empty string
df["overview"] = df["overview"].fillna("")
df["genres"] = df["genres"].fillna("")
df["keywords"] = df["keywords"].fillna("")

# Combine the text-based movie features
df["clean_text"] = (
    df["overview"] + " " +
    df["genres"] + " " +
    df["keywords"]
)

# Convert to lowercase
df["clean_text"] = df["clean_text"].str.lower()

# Remove punctuation and special characters
df["clean_text"] = df["clean_text"].apply(
    lambda x: re.sub(r"[^a-z0-9\s]", " ", x)
)

# Remove extra spaces
df["clean_text"] = df["clean_text"].str.replace(r"\s+", " ", regex=True).str.strip()

# Remove common English stopwords using scikit-learn
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

df["clean_text"] = df["clean_text"].apply(
    lambda x: " ".join(word for word in x.split()
                       if word not in ENGLISH_STOP_WORDS)
)

print(df[["title", "clean_text"]].head())


                                      title  \
0                                    Avatar   
1  Pirates of the Caribbean: At World's End   
2                                   Spectre   
3                     The Dark Knight Rises   
4                               John Carter   

                                          clean_text  
0  22nd century paraplegic marine dispatched moon...  
1  captain barbossa long believed dead come life ...  
2  cryptic message bond s past sends trail uncove...  
3  following death district attorney harvey dent ...  
4  john carter war weary military captain s inexp...  


### Task3 TF-IDF Vectorization


In [6]:
# Create TF-IDF vectorizer
tfidf = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2)
)

# Convert cleaned text into TF-IDF vectors
tfidf_matrix = tfidf.fit_transform(df["clean_text"])

print("TF-IDF Matrix Shape:")
print(tfidf_matrix.shape)

print("\nNumber of Features:")
print(len(tfidf.get_feature_names_out()))

print("\nSample Features:")
print(tfidf.get_feature_names_out()[:20])


TF-IDF Matrix Shape:
(4803, 5000)

Number of Features:
5000

Sample Features:
['000' '10' '10 year' '100' '1003 photographer' '10041'
 '10041 dysfunctional' '10051' '10051 heist' '10084' '10084 rescue'
 '10085' '10085 betrayal' '1009' '1009 baby' '10093' '10093 priest' '1010'
 '1010 bar' '10103']


### Task4 Similarity Computation


In [7]:
# Compute cosine similarity between all movies
similarity_matrix = cosine_similarity(tfidf_matrix)

print("Similarity Matrix Shape:")
print(similarity_matrix.shape)

print("\nSample Similarity Values:")
print(similarity_matrix[:5, :5])

# Cosine similarity is used because it measures the angle/direction
# between two TF-IDF vectors and works well with sparse text data.


Similarity Matrix Shape:
(4803, 4803)

Sample Similarity Values:
[[1.         0.18591999 0.11346435 0.15545868 0.41270097]
 [0.18591999 1.         0.10662484 0.1497819  0.16734635]
 [0.11346435 0.10662484 1.         0.11390327 0.14673992]
 [0.15545868 0.1497819  0.11390327 1.         0.13235786]
 [0.41270097 0.16734635 0.14673992 0.13235786 1.        ]]


### Task5 Build Recommendation Function


In [8]:
def recommend(item_name, top_n=5):
    # Find the selected movie
    matches = df.index[df["title"].str.lower() == item_name.lower()].tolist()

    if not matches:
        return pd.DataFrame(columns=["title", "similarity_score"])

    item_index = matches[0]

    # Get similarity scores for the selected movie
    scores = list(enumerate(similarity_matrix[item_index]))

    # Sort from highest to lowest similarity
    scores = sorted(scores, key=lambda x: x[1], reverse=True)

    # Remove the selected movie itself
    scores = [item for item in scores if item[0] != item_index]

    # Return top N recommendations
    top_items = scores[:top_n]

    recommendations = pd.DataFrame({
        "title": [df.iloc[i]["title"] for i, score in top_items],
        "similarity_score": [score for i, score in top_items]
    })

    return recommendations

# Test with at least 3 different movies
test_items = df["title"].dropna().drop_duplicates().head(3).tolist()

for item in test_items:
    print("\nRecommendations for:", item)
    print(recommend(item, top_n=5))



Recommendations for: Avatar
                title  similarity_score
0              Aliens          0.461133
1              Alien³          0.452970
2  Planet of the Apes          0.445253
3            Galaxina          0.439247
4      Silent Running          0.437439

Recommendations for: Pirates of the Caribbean: At World's End
                                               title  similarity_score
0         Pirates of the Caribbean: Dead Man's Chest          0.527463
1                                       Nim's Island          0.441742
2                                         Life of Pi          0.424403
3  Pirates of the Caribbean: The Curse of the Bla...          0.421417
4                                   Cutthroat Island          0.383098

Recommendations for: Spectre
                   title  similarity_score
0      Quantum of Solace          0.528316
1                Skyfall          0.523244
2  Never Say Never Again          0.512622
3        Die Another Day          0.5121